# Notebook for the 1000 Runs Ensemble

In [1]:
import pandas as pd
import os
from utils.eda_utils import EDAUtils
import boto3

In [ ]:
%load_ext autoreload
%autoreload 2

In [2]:
SCRIPT_DIR_PATH = os.getcwd()
ROOT_DIR_PATH = os.path.dirname(SCRIPT_DIR_PATH)
DATA_DIR_PATH = os.path.join(ROOT_DIR_PATH, "data")
MAPPING_DIR_PATH = os.path.join(DATA_DIR_PATH, "mapping")
SSP_DIR_PATH = os.path.join(DATA_DIR_PATH, "ssp")
TRAINING_DIR_PATH = os.path.join(DATA_DIR_PATH, "training")
CONFIG_DIR_PATH = os.path.join(ROOT_DIR_PATH, "config")

In [ ]:
os.makedirs(SSP_DIR_PATH, exist_ok=True)

In [3]:
edau = EDAUtils()

## Pull data from AWS S3


In [ ]:
aws_config = edau.read_yaml(os.path.join(CONFIG_DIR_PATH, "aws_credentials_config.yaml"))
profile_name = aws_config["profile_name"]
bucket_name = aws_config["bucket_name"]
# Set your profile
session = boto3.Session(profile_name=profile_name)

# Create an S3 client or resource
s3 = session.resource('s3')

# Define folder prefix
prefix = 'sisepuede_summary_results_run_sisepuede_run_2025-08-10t10;29;30.545790/'  # this is like the "folder" in S3

In [ ]:
# Local destination
destination = os.path.join(SSP_DIR_PATH, prefix.strip('/'))
if os.path.exists(destination) and os.listdir(destination):
    print(f"Destination '{destination}' already exists and is not empty. Skipping download.")
else:
    os.makedirs(destination, exist_ok=True)
    bucket = s3.Bucket(bucket_name)
    for obj in bucket.objects.filter(Prefix=prefix):
        if obj.key.endswith('/'):  # skip directories
            continue
        target_path = os.path.join(destination, os.path.basename(obj.key))
        bucket.download_file(obj.key, target_path)
        print(f"Downloaded: {obj.key}")

In [4]:
# Define folder prefix
prefix = 'sisepuede_summary_results_run_sisepuede_run_2025-08-10t10;29;30.545790/' 

In [5]:
SIMULATION_DIR_PATH = os.path.join(SSP_DIR_PATH, prefix.strip('/'))

## Load and Process LHC Samples Dataframes

In [6]:
# Load lhc samples dfs
lhs_exogenous_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_EXOGENOUS_UNCERTAINTIES.csv"))
lhs_levers_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_LHC_SAMPLES_LEVER_EFFECTS.csv"))

In [ ]:
lhs_exogenous_df.head()

In [ ]:
lhs_levers_df.head()

In [ ]:
lhs_exogenous_df.info()

In [ ]:
lhs_levers_df.info()

In [7]:
lhs_df_merged = pd.merge(lhs_exogenous_df, lhs_levers_df, on=["region", "design_id", "future_id"], how="outer", suffixes=('_X', '_L'))
lhs_df_merged.head()

,region,design_id,future_id,47,48,49,50,51,52,53,...,1739,1740,1745,1746,1748,1756,1758,1770,1771,1779
0,louisiana,-1,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,...,0.367236,0.680655,0.917957,0.134558,0.193991,0.644576,0.214517,0.955397,0.733289,0.116270
1,louisiana,-1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,...,0.840756,0.705425,0.984197,0.409837,0.514426,0.486411,0.278740,0.680327,0.626129,0.392241
2,louisiana,-1,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,...,0.691967,0.816298,0.709269,0.917388,0.682384,0.331853,0.193970,0.968824,0.467563,0.402772
3,louisiana,-1,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,...,0.748838,0.617573,0.971575,0.477516,0.762533,0.267855,0.674344,0.246455,0.464470,0.968928
4,louisiana,-1,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,...,0.942745,0.207604,0.936352,0.371713,0.712287,0.135855,0.111477,0.455590,0.378535,0.267191


In [ ]:
# NOTE: check col names, there should be no duplicates
lhs_df_merged.columns

In [ ]:
lhs_df_merged.info()

## Load SISEPUEDE WIDE_INPUTS_OUTPUTS

In [8]:
wide_inputs_outputs_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "sisepuede_results_sisepuede_run_2025-08-10t10;29;30.545790_IDE_WIDE_INPUTS_OUTPUTS.csv"))
wide_inputs_outputs_df

,Index,time_period,primary_id,region,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,...,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu
0,louisiana_354660,7,354660,louisiana,0.0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,...,1.233181,3.138402,0.447204,0.000000,30.469259,13.094104,117.042622,4.491483,45.130223,2.629950
1,louisiana_354660,8,354660,louisiana,0.0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,...,1.227027,3.190377,0.453932,0.000000,38.785521,15.448650,114.205980,4.519073,46.068411,2.641921
2,louisiana_354660,9,354660,louisiana,0.0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,...,1.210373,3.241353,0.460975,0.000000,35.280608,14.868080,114.046420,4.549158,47.114101,2.656241
3,louisiana_354660,10,354660,louisiana,0.0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,...,1.181755,3.293836,0.468262,0.000000,47.213230,17.172010,112.604235,4.581895,48.249651,2.672838
4,louisiana_354660,11,354660,louisiana,0.0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,...,1.140940,3.347158,0.475743,0.000000,47.401636,17.516973,111.117115,4.617352,49.461277,2.691630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28850,louisiana_355313,31,355313,louisiana,0.0,438716.776515,74294.079256,108.359910,101670.222481,9598.215173,...,2.905346,5.297445,0.552432,-1.199164,46.662677,14.159895,62.516491,3.070915,51.396540,2.243853
28851,louisiana_355313,32,355313,louisiana,0.0,443433.665754,75069.590358,109.953198,102787.213521,9740.898133,...,2.913548,5.406589,0.554898,-1.262278,46.851672,13.945969,60.984318,2.970735,51.503706,2.218678
28852,louisiana_355313,33,355313,louisiana,0.0,447981.700430,75825.214323,111.498781,103856.137164,9878.779082,...,2.919315,5.516952,0.557270,-1.325391,47.059094,13.732943,59.489538,2.867692,51.602941,2.192627
28853,louisiana_355313,34,355313,louisiana,0.0,452396.830738,76564.024281,113.008581,104888.418745,10013.113724,...,2.904608,5.628525,0.559544,-1.388505,47.284998,13.520595,58.034030,2.761373,51.691903,2.165547


In [ ]:
wide_inputs_outputs_df['primary_id'].unique()

## Load Costs-Benefits Data

In [9]:
cb_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "wide_cb_data_lhc_2025-08-10t10;29;30.545790.csv"))
cb_df.head()

,primary_id,future_id,strategy_code,Year,air_pollution,congestion,consumer_savings,crop_value,ecosystem_services,env_pollution,...,human_health,ippu_value,land_pollution,lvst_value,road_safety,sector_specific,system_cost,technical_cost,technical_savings,water_pollution
0,354355,1,PFLO:ALL_LA_ACTIONS,2022.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,354355,1,PFLO:ALL_LA_ACTIONS,2023.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,354355,1,PFLO:ALL_LA_ACTIONS,2024.0,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,354355,1,PFLO:ALL_LA_ACTIONS,2025.0,0.060282,0.132495,0.021720,4.143264e-14,-3.977853e-13,0.000014,...,0.018795,-0.053377,-1.193712e-17,-0.131033,0.210159,7536.242379,37.153191,0.619617,0.207821,-0.025037
4,354355,1,PFLO:ALL_LA_ACTIONS,2026.0,0.073026,0.159321,0.043711,-1.172087e-14,4.620699e-13,0.000017,...,0.038597,-0.057632,1.875833e-17,-0.156095,0.252749,9164.677971,45.045727,-0.196941,0.192236,-0.030054


In [10]:
for col in cb_df.columns:
    print(col)

primary_id
future_id
strategy_code
Year
air_pollution
congestion
consumer_savings
crop_value
ecosystem_services
env_pollution
fuel_cost
human_health
ippu_value
land_pollution
lvst_value
road_safety
sector_specific
system_cost
technical_cost
technical_savings
water_pollution


In [ ]:
cb_df.tail()

In [ ]:
for col in cb_df.columns:
    print(col)

## Data Cleaning

### SISEPUEDE Emission data

In [11]:
# Get the subsector total variables
subsector_total_vars = [c for c in wide_inputs_outputs_df.columns if "emission_co2e_subsector_total" in c]

In [12]:
# Filter to only subsector total columns and primary_id, time_period
la_emissions_df = wide_inputs_outputs_df[["primary_id", "time_period"] + subsector_total_vars]
la_emissions_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu
0,354660,7,2.923974,-36.545088,2.050000,0.230319,1.539811,1.233181,3.138402,0.447204,0.0,30.469259,13.094104,117.042622,4.491483,45.130223,2.629950
1,354660,8,2.863991,-39.756859,2.039113,0.229380,1.516608,1.227027,3.190377,0.453932,0.0,38.785521,15.448650,114.205980,4.519073,46.068411,2.641921
2,354660,9,2.911902,-42.274952,2.028288,0.228514,1.493765,1.210373,3.241353,0.460975,0.0,35.280608,14.868080,114.046420,4.549158,47.114101,2.656241
3,354660,10,2.899808,-44.341002,2.017523,0.227722,1.471286,1.181755,3.293836,0.468262,0.0,47.213230,17.172010,112.604235,4.581895,48.249651,2.672838
4,354660,11,2.887734,-46.106391,2.006818,0.226980,1.449157,1.140940,3.347158,0.475743,0.0,47.401636,17.516973,111.117115,4.617352,49.461277,2.691630


In [ ]:
la_emissions_df.tail()

### Production Data

In [13]:
# Get the subsector total variables
industry_value_fuel_vars = [c for c in wide_inputs_outputs_df.columns if "totalvalue_enfu_fuel_consumed_inen" in c]

In [14]:
# Filter to only production columns avoiding "subsector" total columns
industrial_production_df = wide_inputs_outputs_df[["primary_id", "time_period"] + industry_value_fuel_vars]
industrial_production_df

,primary_id,time_period,totalvalue_enfu_fuel_consumed_inen_fuel_biomass,totalvalue_enfu_fuel_consumed_inen_fuel_coal,totalvalue_enfu_fuel_consumed_inen_fuel_coke,totalvalue_enfu_fuel_consumed_inen_fuel_diesel,totalvalue_enfu_fuel_consumed_inen_fuel_electricity,totalvalue_enfu_fuel_consumed_inen_fuel_furnace_gas,totalvalue_enfu_fuel_consumed_inen_fuel_gasoline,totalvalue_enfu_fuel_consumed_inen_fuel_hydrocarbon_gas_liquids,totalvalue_enfu_fuel_consumed_inen_fuel_hydrogen,totalvalue_enfu_fuel_consumed_inen_fuel_kerosene,totalvalue_enfu_fuel_consumed_inen_fuel_natural_gas,totalvalue_enfu_fuel_consumed_inen_fuel_oil
0,354660,7,3.529476,0.392164,170.778994,1170.807186,5.245482e+07,173.924934,0.0,0.323006,0.0,1930.469161,260.150186,10.019687
1,354660,8,0.547550,0.060839,171.212393,1138.600316,5.210628e+07,164.123248,0.0,0.353406,0.0,1655.933926,266.047918,10.081007
2,354660,9,0.217229,0.024137,179.237211,1126.629254,5.101475e+07,165.403061,0.0,0.371192,0.0,1581.765027,276.813846,10.147874
3,354660,10,0.176124,0.019569,175.949057,1101.288787,4.967968e+07,161.070682,0.0,0.395853,0.0,1576.792179,268.130952,10.220654
4,354660,11,0.134588,0.014954,172.235775,1090.243305,4.828760e+07,155.131422,0.0,0.416836,0.0,1576.433394,254.848470,10.299687
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28850,355313,31,0.000000,0.000000,154.377500,1562.616196,2.778289e+07,151.332762,0.0,65.370148,0.0,1665.734539,324.799349,6.899916
28851,355313,32,0.000000,0.000000,152.864724,1598.841754,2.737439e+07,149.909307,0.0,66.542848,0.0,1667.832475,324.073733,6.674668
28852,355313,33,0.000000,0.000000,150.589562,1636.207025,2.700305e+07,148.427696,0.0,67.534953,0.0,1676.250543,321.833389,6.442889
28853,355313,34,0.000000,0.000000,149.100567,1665.581592,2.666349e+07,145.807986,0.0,68.341702,0.0,1677.733590,320.333261,6.203799


### Production Cost Data

In [15]:
# 2) Define your fuels and sectors
relevant_fuels = ['biomass', 
                    'coal', 
                    'coke', 
                    'diesel', 
                    'electricity',
                    'furnace_gas',
                    'gasoline', 
                    'hydrocarbon_gas_liquids',
                    'hydrogen',
                    'kerosene',
                    'natural_gas',
                    'oil']

sectors = ['agriculture_and_livestock',
           'cement',
           'chemicals',
           'electronics',
           'glass',
           'lime_and_carbonite',
           'metals',
           'mining',
           'other_product_manufacturing',
           'paper',
           'plastic',
           'recycled_glass',
           'recycled_metals',
           'recycled_paper',
           'recycled_plastic',
           'recycled_rubber_and_leather',
           'recycled_textiles',
           'recycled_wood',
           'rubber_and_leather',
           'textiles',
           'wood']

In [16]:
# 4) Industrial cost parameters
capex_industrial_electricity = 92666.6 * 21
capex_industrial_other       = 92666.6 * 12
opex_industrial_electricity  = 92666.6 * 2.5
opex_industrial_other        = 92666.6 * 4.5
capex_multiplier_efficiency = 10000000
opex_multiplier_efficiency = 0

In [17]:
# 5) Loop over fuels and sectors
# Use the original dataframe directly
# Initialize results dataframe
ind_fuel_demand_by_sector = pd.DataFrame({
    'primary_id': wide_inputs_outputs_df['primary_id'],
    'time_period': wide_inputs_outputs_df['time_period']
}, index=wide_inputs_outputs_df.index)

# Loop over fuels and sectors
for fuel in relevant_fuels:
    # efficiency column for this fuel
    eff_cols = [c for c in wide_inputs_outputs_df.columns
                if c.startswith(f'efficfactor_enfu_industrial_energy_fuel_{fuel}')]
    if not eff_cols:
        continue
    fuel_efficiency = wide_inputs_outputs_df[eff_cols[0]]

    for sector in sectors:
        sector_dem_cols = [c for c in wide_inputs_outputs_df.columns
                           if f'energy_demand_inen_{sector}' in c]
        sector_fuel_fraction_cols = [c for c in wide_inputs_outputs_df.columns
                                     if f'frac_inen_energy_{sector}_{fuel}' in c]

        if sector_dem_cols and sector_fuel_fraction_cols:
            sector_total_demand = wide_inputs_outputs_df[sector_dem_cols[0]]
            sector_fuel_fraction = wide_inputs_outputs_df[sector_fuel_fraction_cols[0]]

            if (sector_fuel_fraction * sector_total_demand).sum() > 0 or fuel == 'electricity':
                sector_fuel_demand = sector_fuel_fraction * sector_total_demand
                ind_fuel_demand_by_sector[f'energy_demand_{sector}_{fuel}'] = sector_fuel_demand

                # CAPEX/OPEX
                if fuel == 'electricity':
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_electricity
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_electricity
                    )
                else:
                    ind_fuel_demand_by_sector[f'energy_demand_capex_{sector}_{fuel}'] = (
                        sector_fuel_demand * capex_industrial_other
                    )
                    ind_fuel_demand_by_sector[f'energy_demand_opex_{sector}_{fuel}'] = (
                        sector_fuel_demand * opex_industrial_other
                    )

                # Fuel consumed
                sector_fuel_consumed = sector_fuel_demand / fuel_efficiency

                # Baseline = first time_period per primary_id
                sector_fuel_consumed_baseline = (
                    sector_fuel_consumed.groupby(wide_inputs_outputs_df['primary_id'])
                                        .transform('first')
                )

                # Change relative to baseline
                sector_change_in_fuel_consumed = (
                    sector_fuel_consumed_baseline - sector_fuel_consumed
                )

                # Save results
                ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed
                )
                ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * capex_multiplier_efficiency
                )
                ind_fuel_demand_by_sector[f'efficiency_opex_{sector}_{fuel}'] = (
                    sector_change_in_fuel_consumed * opex_multiplier_efficiency
                )


C:\Users\nasta\AppData\Local\Temp\ipykernel_209488\3509172589.py:63: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_energy_saving_{sector}_{fuel}'] = (
C:\Users\nasta\AppData\Local\Temp\ipykernel_209488\3509172589.py:66: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  ind_fuel_demand_by_sector[f'efficiency_capex_{sector}_{fuel}'] = (
C:\Users\nasta\AppData\Local\Temp\ipykernel_209488\3509172589.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `f

In [18]:
ind_fuel_demand_by_sector

,primary_id,time_period,energy_demand_cement_coal,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,efficiency_energy_saving_cement_coal,efficiency_capex_cement_coal,efficiency_opex_cement_coal,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,...,energy_demand_opex_recycled_wood_oil,efficiency_energy_saving_recycled_wood_oil,efficiency_capex_recycled_wood_oil,efficiency_opex_recycled_wood_oil,energy_demand_rubber_and_leather_oil,energy_demand_capex_rubber_and_leather_oil,energy_demand_opex_rubber_and_leather_oil,efficiency_energy_saving_rubber_and_leather_oil,efficiency_capex_rubber_and_leather_oil,efficiency_opex_rubber_and_leather_oil
0,354660,7,15.093578,1.678405e+07,6.294018e+06,0.000000,0.000000e+00,0.0,0.000731,813.000109,...,134043.774775,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,354660,8,12.273835,1.364849e+07,5.118186e+06,4.762397,4.762397e+07,0.0,0.000070,77.401644,...,132036.505743,0.008078,8.077733e+04,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,354660,9,11.819088,1.314282e+07,4.928556e+06,5.592567,5.592567e+07,0.0,0.000000,0.000000,...,129034.828537,0.019239,1.923906e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,354660,10,11.584514,1.288197e+07,4.830739e+06,6.055299,6.055299e+07,0.0,0.000000,0.000000,...,125949.483810,0.030581,3.058090e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,354660,11,10.749737,1.195370e+07,4.482637e+06,7.495087,7.495087e+07,0.0,0.000000,0.000000,...,123239.783173,0.040661,4.066115e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28850,355313,31,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,38307.074521,0.327810,3.278102e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28851,355313,32,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,35811.489347,0.335051,3.350506e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28852,355313,33,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,33430.178820,0.341880,3.418804e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0
28853,355313,34,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,31154.629122,0.348332,3.483322e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#example= ind_fuel_demand_by_sector[ind_fuel_demand_by_sector['primary_id'].isin([354660])]

In [ ]:
#example

### CB Data

In [19]:
# Make all column names lowercase
cb_df.columns = [c.lower() for c in cb_df.columns]

# Filter to only important cb columns
cb_df = cb_df[["primary_id",
               "future_id",
               "year",
               "technical_cost",
               "consumer_savings",
               "human_health",
               "air_pollution"]]

cb_df.head()

,primary_id,future_id,year,technical_cost,consumer_savings,human_health,air_pollution
0,354355,1,2022.0,0.000000,0.000000,0.000000,0.000000
1,354355,1,2023.0,0.000000,0.000000,0.000000,0.000000
2,354355,1,2024.0,0.000000,0.000000,0.000000,0.000000
3,354355,1,2025.0,0.619617,0.021720,0.018795,0.060282
4,354355,1,2026.0,-0.196941,0.043711,0.038597,0.073026


In [20]:
cb_df ['primary_id'].unique()

array([354355, 354356, 354357, 354358, 354359, 354360, 354361, 354362,
       354363, 354364, 354365, 354366, 354367, 354368, 354369, 354370,
       354371, 354372, 354373, 354374, 354375, 354376, 354377, 354378,
       354379, 354380, 354381, 354382, 354383, 354384, 354385, 354386,
       354387, 354388, 354389, 354390, 354391, 354392, 354393, 354394,
       354395, 354396, 354397, 354398, 354399, 354400, 354401, 354402,
       354403, 354404, 354405, 354406, 354407, 354408, 354409, 354410,
       354411, 354412, 354413, 354414, 354415, 354416, 354417, 354418,
       354419, 354420, 354421, 354422, 354423, 354424, 354425, 354426,
       354427, 354428, 354429, 354430, 354431, 354432, 354433, 354434,
       354435, 354436, 354437, 354438, 354439, 354440, 354441, 354442,
       354443, 354444, 354445, 354446, 354447, 354448, 354449, 354450,
       354451, 354452, 354453, 354454, 354455, 354456, 354457, 354458,
       354459, 354460, 354461, 354462, 354463, 354464, 354465, 354466,
      

In [ ]:
cb_df.tail()

In [ ]:
cb_df.info()

## LHS Data

In [21]:
lhs_df_merged = lhs_df_merged.drop(columns=["design_id", "region"])
lhs_df_merged.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1739,1740,1745,1746,1748,1756,1758,1770,1771,1779
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.367236,0.680655,0.917957,0.134558,0.193991,0.644576,0.214517,0.955397,0.733289,0.116270
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.840756,0.705425,0.984197,0.409837,0.514426,0.486411,0.278740,0.680327,0.626129,0.392241
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.691967,0.816298,0.709269,0.917388,0.682384,0.331853,0.193970,0.968824,0.467563,0.402772
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.748838,0.617573,0.971575,0.477516,0.762533,0.267855,0.674344,0.246455,0.464470,0.968928
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.942745,0.207604,0.936352,0.371713,0.712287,0.135855,0.111477,0.455590,0.378535,0.267191


## Transform time series format into single-row format

### SISEPUEDE Emission data

In [22]:
# Sum all the subsector emission columns across axis=1
la_emission_total_df = la_emissions_df.copy()
la_emission_total_df["emission_total"] = la_emission_total_df[subsector_total_vars].sum(axis=1)
la_emission_total_df.head()

,primary_id,time_period,emission_co2e_subsector_total_agrc,emission_co2e_subsector_total_frst,emission_co2e_subsector_total_lndu,emission_co2e_subsector_total_lsmm,emission_co2e_subsector_total_lvst,emission_co2e_subsector_total_soil,emission_co2e_subsector_total_waso,emission_co2e_subsector_total_trww,emission_co2e_subsector_total_ccsq,emission_co2e_subsector_total_entc,emission_co2e_subsector_total_fgtv,emission_co2e_subsector_total_inen,emission_co2e_subsector_total_scoe,emission_co2e_subsector_total_trns,emission_co2e_subsector_total_ippu,emission_total
0,354660,7,2.923974,-36.545088,2.050000,0.230319,1.539811,1.233181,3.138402,0.447204,0.0,30.469259,13.094104,117.042622,4.491483,45.130223,2.629950,187.875442
1,354660,8,2.863991,-39.756859,2.039113,0.229380,1.516608,1.227027,3.190377,0.453932,0.0,38.785521,15.448650,114.205980,4.519073,46.068411,2.641921,193.433125
2,354660,9,2.911902,-42.274952,2.028288,0.228514,1.493765,1.210373,3.241353,0.460975,0.0,35.280608,14.868080,114.046420,4.549158,47.114101,2.656241,187.814826
3,354660,10,2.899808,-44.341002,2.017523,0.227722,1.471286,1.181755,3.293836,0.468262,0.0,47.213230,17.172010,112.604235,4.581895,48.249651,2.672838,199.713047
4,354660,11,2.887734,-46.106391,2.006818,0.226980,1.449157,1.140940,3.347158,0.475743,0.0,47.401636,17.516973,111.117115,4.617352,49.461277,2.691630,198.234123


In [23]:
# Keep only the primary_id, time_period, and emission_total columns
la_emission_total_df = la_emission_total_df[["primary_id", "time_period", "emission_total"]]
la_emission_total_df.head()

,primary_id,time_period,emission_total
0,354660,7,187.875442
1,354660,8,193.433125
2,354660,9,187.814826
3,354660,10,199.713047
4,354660,11,198.234123


In [ ]:
la_emission_total_df.tail()

### Emission data sum

In [24]:
# aggregate data by primary_id summing the emissions
la_emission_df_sum_agg = la_emission_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_emission_df_sum_agg = la_emission_df_sum_agg.drop(columns=["time_period"])
la_emission_df_sum_agg.head()

,primary_id,emission_total
0,354354,1700.012557
1,354355,3553.270010
2,354356,3028.607700
3,354357,4127.536839
4,354358,2668.240547


### Emission data mean

In [25]:
# Filter out rows with time_period < 31
la_filtered_emission_total_df = la_emission_total_df[la_emission_total_df["time_period"] >= 31]
la_filtered_emission_total_df = la_filtered_emission_total_df.reset_index(drop=True)
la_filtered_emission_total_df.head(7)

,primary_id,time_period,emission_total
0,354660,31,53.155458
1,354660,32,49.182152
2,354660,33,45.371484
3,354660,34,41.661921
4,354660,35,38.180330
5,354533,31,24.009496
6,354533,32,19.425322


In [26]:
# aggregate data by primary_id by summing the emissions
la_emission_df_mean_agg = la_filtered_emission_total_df.groupby(["primary_id"]).mean().reset_index()

# Rename emission_total to emission_avg_last_five_years
la_emission_df_mean_agg.rename(columns={"emission_total": "emission_avg_last_five_years"}, inplace=True)
la_emission_df_mean_agg

,primary_id,time_period,emission_avg_last_five_years
0,354354,33.0,-54.039699
1,354355,33.0,59.720047
2,354356,33.0,30.972144
3,354357,33.0,92.551550
4,354358,33.0,11.015231
...,...,...,...
990,355350,33.0,23.599187
991,355351,33.0,60.727431
992,355352,33.0,51.497226
993,355353,33.0,62.552429


In [27]:
# Drop year column as it is no longer needed
la_emission_df_mean_agg = la_emission_df_mean_agg.drop(columns=["time_period"])
la_emission_df_mean_agg.head()

,primary_id,emission_avg_last_five_years
0,354354,-54.039699
1,354355,59.720047
2,354356,30.972144
3,354357,92.551550
4,354358,11.015231


### Combining emission agg into one df

In [28]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)
print("la_emission_df_sum_agg shape:", la_emission_df_sum_agg.shape)

la_emission_df_mean_agg shape: (995, 2)
la_emission_df_sum_agg shape: (995, 2)


In [29]:
la_emissions_df_merged = la_emission_df_mean_agg.merge(la_emission_df_sum_agg, on="primary_id", how="inner")
la_emissions_df_merged.head()

,primary_id,emission_avg_last_five_years,emission_total
0,354354,-54.039699,1700.012557
1,354355,59.720047,3553.270010
2,354356,30.972144,3028.607700
3,354357,92.551550,4127.536839
4,354358,11.015231,2668.240547


In [30]:
print("la_emission_df_mean_agg shape:", la_emission_df_mean_agg.shape)

la_emission_df_mean_agg shape: (995, 2)


### Production Data

In [31]:
# Sum all the subsector emission columns across axis=1
la_production_total_df = industrial_production_df.copy()
la_production_total_df["production_total"] = la_production_total_df[industry_value_fuel_vars].sum(axis=1)
la_production_total_df.head()

,primary_id,time_period,totalvalue_enfu_fuel_consumed_inen_fuel_biomass,totalvalue_enfu_fuel_consumed_inen_fuel_coal,totalvalue_enfu_fuel_consumed_inen_fuel_coke,totalvalue_enfu_fuel_consumed_inen_fuel_diesel,totalvalue_enfu_fuel_consumed_inen_fuel_electricity,totalvalue_enfu_fuel_consumed_inen_fuel_furnace_gas,totalvalue_enfu_fuel_consumed_inen_fuel_gasoline,totalvalue_enfu_fuel_consumed_inen_fuel_hydrocarbon_gas_liquids,totalvalue_enfu_fuel_consumed_inen_fuel_hydrogen,totalvalue_enfu_fuel_consumed_inen_fuel_kerosene,totalvalue_enfu_fuel_consumed_inen_fuel_natural_gas,totalvalue_enfu_fuel_consumed_inen_fuel_oil,production_total
0,354660,7,3.529476,0.392164,170.778994,1170.807186,5.245482e+07,173.924934,0.0,0.323006,0.0,1930.469161,260.150186,10.019687,5.245854e+07
1,354660,8,0.547550,0.060839,171.212393,1138.600316,5.210628e+07,164.123248,0.0,0.353406,0.0,1655.933926,266.047918,10.081007,5.210969e+07
2,354660,9,0.217229,0.024137,179.237211,1126.629254,5.101475e+07,165.403061,0.0,0.371192,0.0,1581.765027,276.813846,10.147874,5.101809e+07
3,354660,10,0.176124,0.019569,175.949057,1101.288787,4.967968e+07,161.070682,0.0,0.395853,0.0,1576.792179,268.130952,10.220654,4.968298e+07
4,354660,11,0.134588,0.014954,172.235775,1090.243305,4.828760e+07,155.131422,0.0,0.416836,0.0,1576.433394,254.848470,10.299687,4.829086e+07


In [32]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_total_df = la_production_total_df[["primary_id", "time_period", "production_total"]]
la_production_total_df.head()

,primary_id,time_period,production_total
0,354660,7,5.245854e+07
1,354660,8,5.210969e+07
2,354660,9,5.101809e+07
3,354660,10,4.968298e+07
4,354660,11,4.829086e+07


In [33]:
# aggregate data by primary_id summing the emissions
la_production_df_sum_agg = la_production_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_production_df_sum_agg = la_production_df_sum_agg.drop(columns=["time_period"])
la_production_df_sum_agg.head()

,primary_id,production_total
0,354354,6.787144e+08
1,354355,7.915055e+08
2,354356,8.012644e+08
3,354357,9.337928e+08
4,354358,8.241354e+08


### Production Cost Data

In [34]:
industry_cost_vars = [
    c for c in ind_fuel_demand_by_sector.columns
    if c.startswith("energy_demand_capex_") or c.startswith("energy_demand_opex_")
]

In [35]:
ind_fuel_demand_by_sector[industry_cost_vars]

,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,energy_demand_capex_chemicals_coal,energy_demand_opex_chemicals_coal,energy_demand_capex_glass_coal,energy_demand_opex_glass_coal,energy_demand_capex_lime_and_carbonite_coal,energy_demand_opex_lime_and_carbonite_coal,energy_demand_capex_metals_coal,energy_demand_opex_metals_coal,...,energy_demand_capex_recycled_glass_oil,energy_demand_opex_recycled_glass_oil,energy_demand_capex_recycled_metals_oil,energy_demand_opex_recycled_metals_oil,energy_demand_capex_recycled_rubber_and_leather_oil,energy_demand_opex_recycled_rubber_and_leather_oil,energy_demand_capex_recycled_wood_oil,energy_demand_opex_recycled_wood_oil,energy_demand_capex_rubber_and_leather_oil,energy_demand_opex_rubber_and_leather_oil
0,1.678405e+07,6.294018e+06,813.000109,304.875041,0.0,0.0,0.0,0.0,829010.070448,310878.776418,...,0.0,0.0,0.0,0.0,0.0,0.0,357450.066066,134043.774775,0.0,0.0
1,1.364849e+07,5.118186e+06,77.401644,29.025617,0.0,0.0,0.0,0.0,817483.443106,306556.291165,...,0.0,0.0,0.0,0.0,0.0,0.0,352097.348647,132036.505743,0.0,0.0
2,1.314282e+07,4.928556e+06,0.000000,0.000000,0.0,0.0,0.0,0.0,808226.548542,303084.955703,...,0.0,0.0,0.0,0.0,0.0,0.0,344092.876099,129034.828537,0.0,0.0
3,1.288197e+07,4.830739e+06,0.000000,0.000000,0.0,0.0,0.0,0.0,794417.520479,297906.570180,...,0.0,0.0,0.0,0.0,0.0,0.0,335865.290159,125949.483810,0.0,0.0
4,1.195370e+07,4.482637e+06,0.000000,0.000000,0.0,0.0,0.0,0.0,781085.518864,292907.069574,...,0.0,0.0,0.0,0.0,0.0,0.0,328639.421796,123239.783173,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28850,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.0,0.0,572217.863629,214581.698861,...,0.0,0.0,0.0,0.0,0.0,0.0,102152.198723,38307.074521,0.0,0.0
28851,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.0,0.0,568737.278502,213276.479438,...,0.0,0.0,0.0,0.0,0.0,0.0,95497.304925,35811.489347,0.0,0.0
28852,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.0,0.0,565726.633274,212147.487478,...,0.0,0.0,0.0,0.0,0.0,0.0,89147.143520,33430.178820,0.0,0.0
28853,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.0,0.0,0.0,563161.897101,211185.711413,...,0.0,0.0,0.0,0.0,0.0,0.0,83079.010992,31154.629122,0.0,0.0


In [36]:
# Sum all the subsector emission columns across axis=1
la_production_cost_total_df = ind_fuel_demand_by_sector.copy()
la_production_cost_total_df["production_cost_total"] = la_production_cost_total_df[industry_cost_vars].sum(axis=1)
la_production_cost_total_df.head()

,primary_id,time_period,energy_demand_cement_coal,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,efficiency_energy_saving_cement_coal,efficiency_capex_cement_coal,efficiency_opex_cement_coal,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,...,efficiency_energy_saving_recycled_wood_oil,efficiency_capex_recycled_wood_oil,efficiency_opex_recycled_wood_oil,energy_demand_rubber_and_leather_oil,energy_demand_capex_rubber_and_leather_oil,energy_demand_opex_rubber_and_leather_oil,efficiency_energy_saving_rubber_and_leather_oil,efficiency_capex_rubber_and_leather_oil,efficiency_opex_rubber_and_leather_oil,production_cost_total
0,354660,7,15.093578,1.678405e+07,6.294018e+06,0.000000,0.000000e+00,0.0,0.000731,813.000109,...,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.615177e+08
1,354660,8,12.273835,1.364849e+07,5.118186e+06,4.762397,4.762397e+07,0.0,0.000070,77.401644,...,0.008078,80777.334467,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.565878e+08
2,354660,9,11.819088,1.314282e+07,4.928556e+06,5.592567,5.592567e+07,0.0,0.000000,0.000000,...,0.019239,192390.592204,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.524159e+08
3,354660,10,11.584514,1.288197e+07,4.830739e+06,6.055299,6.055299e+07,0.0,0.000000,0.000000,...,0.030581,305809.018698,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.493140e+08
4,354660,11,10.749737,1.195370e+07,4.482637e+06,7.495087,7.495087e+07,0.0,0.000000,0.000000,...,0.040661,406611.530538,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.467279e+08


In [37]:
la_production_cost_total_df

,primary_id,time_period,energy_demand_cement_coal,energy_demand_capex_cement_coal,energy_demand_opex_cement_coal,efficiency_energy_saving_cement_coal,efficiency_capex_cement_coal,efficiency_opex_cement_coal,energy_demand_chemicals_coal,energy_demand_capex_chemicals_coal,...,efficiency_energy_saving_recycled_wood_oil,efficiency_capex_recycled_wood_oil,efficiency_opex_recycled_wood_oil,energy_demand_rubber_and_leather_oil,energy_demand_capex_rubber_and_leather_oil,energy_demand_opex_rubber_and_leather_oil,efficiency_energy_saving_rubber_and_leather_oil,efficiency_capex_rubber_and_leather_oil,efficiency_opex_rubber_and_leather_oil,production_cost_total
0,354660,7,15.093578,1.678405e+07,6.294018e+06,0.000000,0.000000e+00,0.0,0.000731,813.000109,...,0.000000,0.000000e+00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.615177e+08
1,354660,8,12.273835,1.364849e+07,5.118186e+06,4.762397,4.762397e+07,0.0,0.000070,77.401644,...,0.008078,8.077733e+04,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.565878e+08
2,354660,9,11.819088,1.314282e+07,4.928556e+06,5.592567,5.592567e+07,0.0,0.000000,0.000000,...,0.019239,1.923906e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.524159e+08
3,354660,10,11.584514,1.288197e+07,4.830739e+06,6.055299,6.055299e+07,0.0,0.000000,0.000000,...,0.030581,3.058090e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.493140e+08
4,354660,11,10.749737,1.195370e+07,4.482637e+06,7.495087,7.495087e+07,0.0,0.000000,0.000000,...,0.040661,4.066115e+05,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.467279e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28850,355313,31,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,0.327810,3.278102e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.206911e+08
28851,355313,32,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,0.335051,3.350506e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.314206e+08
28852,355313,33,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,0.341880,3.418804e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.429505e+08
28853,355313,34,0.000000,0.000000e+00,0.000000e+00,25.005557,2.500556e+08,0.0,0.000000,0.000000,...,0.348332,3.483322e+06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.552951e+08


In [38]:
# Keep only the primary_id, time_period, and emission_total columns
la_production_cost_total_df = la_production_cost_total_df[["primary_id", "time_period", "production_cost_total"]]
la_production_cost_total_df.head()

,primary_id,time_period,production_cost_total
0,354660,7,4.615177e+08
1,354660,8,4.565878e+08
2,354660,9,4.524159e+08
3,354660,10,4.493140e+08
4,354660,11,4.467279e+08


In [39]:
# aggregate data by primary_id summing the emissions
la_production_cost_df_sum_agg = la_production_cost_total_df.groupby(["primary_id"]).sum().reset_index()

# Drop time period column
la_production_cost_df_sum_agg = la_production_cost_df_sum_agg.drop(columns=["time_period"])
la_production_cost_df_sum_agg.head()

,primary_id,production_cost_total
0,354354,1.306324e+10
1,354355,1.484591e+10
2,354356,1.451503e+10
3,354357,1.274256e+10
4,354358,1.235943e+10


### CB data

In [40]:
# aggregate data by primary_id and region by summing the technical cost
cb_df_agg = cb_df.groupby(["primary_id", "future_id"]).sum().reset_index()
cb_df_agg

,primary_id,future_id,year,technical_cost,consumer_savings,human_health,air_pollution
0,354355,1,59044.0,-42.336438,5.511622,7.444526,-33.250075
1,354356,2,59044.0,-21.656242,6.947874,8.685293,-19.779789
2,354357,3,59044.0,-40.157702,6.355524,8.469585,-36.090181
3,354358,4,59044.0,-53.155978,7.441004,8.909666,-6.495959
4,354359,5,59044.0,-77.798140,6.063635,6.719518,-42.784060
...,...,...,...,...,...,...,...
989,355350,996,59044.0,-51.800498,8.239335,8.528071,-18.572578
990,355351,997,59044.0,-23.938700,5.442270,6.948918,-19.480756
991,355352,998,59044.0,-129.619974,6.710947,7.131005,-29.458703
992,355353,999,59044.0,-132.245620,6.000841,8.398696,-31.421014


In [41]:
# Drop year column as it is no longer needed
cb_df_agg = cb_df_agg.drop(columns=["year"], errors='ignore')
cb_df_agg.head()

,primary_id,future_id,technical_cost,consumer_savings,human_health,air_pollution
0,354355,1,-42.336438,5.511622,7.444526,-33.250075
1,354356,2,-21.656242,6.947874,8.685293,-19.779789
2,354357,3,-40.157702,6.355524,8.469585,-36.090181
3,354358,4,-53.155978,7.441004,8.909666,-6.495959
4,354359,5,-77.798140,6.063635,6.719518,-42.784060


In [ ]:
cb_df_agg.info()

## Merge emissions and cb data with lhs samples

In [42]:
attr_primary_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "ATTRIBUTE_PRIMARY.csv"))
attr_primary_df

,primary_id,design_id,strategy_id,future_id
0,354354,4,6004,0
1,354355,4,6004,1
2,354356,4,6004,2
3,354357,4,6004,3
4,354358,4,6004,4
...,...,...,...,...
996,355350,4,6004,996
997,355351,4,6004,997
998,355352,4,6004,998
999,355353,4,6004,999


In [43]:
# check for duplicates primary_id
duplicates = attr_primary_df[attr_primary_df.duplicated(subset=["primary_id"], keep=False)]
if not duplicates.empty:
    print("Duplicated primary_id found:")
    print(duplicates)
else:
    print("No duplicated primary_id found.")

No duplicated primary_id found.


In [44]:
la_emission_df_w_future_id = la_emissions_df_merged.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_emission_df_w_future_id = la_emission_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_emission_df_w_future_id

,primary_id,emission_avg_last_five_years,emission_total,future_id
0,354354,-54.039699,1700.012557,0
1,354355,59.720047,3553.270010,1
2,354356,30.972144,3028.607700,2
3,354357,92.551550,4127.536839,3
4,354358,11.015231,2668.240547,4
...,...,...,...,...
990,355350,23.599187,2958.134476,996
991,355351,60.727431,3526.674139,997
992,355352,51.497226,3434.786875,998
993,355353,62.552429,3695.679453,999


In [45]:
la_production_df_w_future_id = la_production_df_sum_agg.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_production_df_w_future_id = la_production_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_production_df_w_future_id

,primary_id,production_total,future_id
0,354354,6.787144e+08,0
1,354355,7.915055e+08,1
2,354356,8.012644e+08,2
3,354357,9.337928e+08,3
4,354358,8.241354e+08,4
...,...,...,...
990,355350,8.786512e+08,996
991,355351,8.906465e+08,997
992,355352,7.832349e+08,998
993,355353,9.307686e+08,999


In [46]:
la_production_cost_df_w_future_id = la_production_cost_df_sum_agg.merge(attr_primary_df, on="primary_id", how="inner")

# Drop design_id and stratgy_id columns
la_production_cost_df_w_future_id = la_production_cost_df_w_future_id.drop(columns=["design_id", "strategy_id"])
la_production_cost_df_w_future_id

,primary_id,production_cost_total,future_id
0,354354,1.306324e+10,0
1,354355,1.484591e+10,1
2,354356,1.451503e+10,2
3,354357,1.274256e+10,3
4,354358,1.235943e+10,4
...,...,...,...
990,355350,1.472198e+10,996
991,355351,1.331408e+10,997
992,355352,1.329021e+10,998
993,355353,1.330764e+10,999


In [47]:
emission_production_w_future_id=pd.merge(la_production_df_w_future_id, la_emission_df_w_future_id, on=["future_id", "primary_id"], how="inner")

In [48]:
emission_production_cost_w_future_id=pd.merge(la_production_cost_df_w_future_id, emission_production_w_future_id, on=["future_id", "primary_id"], how="inner")

In [ ]:
emission_production_cost_w_future_id

In [ ]:
lhs_df_merged

In [49]:
lhs_emissions_merged_df = pd.merge(lhs_df_merged, emission_production_cost_w_future_id, on="future_id", how="inner")
lhs_emissions_merged_df.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1756,1758,1770,1771,1779,primary_id,production_cost_total,production_total,emission_avg_last_five_years,emission_total
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.644576,0.214517,0.955397,0.733289,0.116270,354355,1.484591e+10,7.915055e+08,59.720047,3553.270010
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.486411,0.278740,0.680327,0.626129,0.392241,354356,1.451503e+10,8.012644e+08,30.972144,3028.607700
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.331853,0.193970,0.968824,0.467563,0.402772,354357,1.274256e+10,9.337928e+08,92.551550,4127.536839
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.267855,0.674344,0.246455,0.464470,0.968928,354358,1.235943e+10,8.241354e+08,11.015231,2668.240547
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.135855,0.111477,0.455590,0.378535,0.267191,354359,1.512143e+10,7.863009e+08,41.876734,3343.173022


In [50]:
complete_merged_df = pd.merge(lhs_emissions_merged_df, cb_df_agg, on=["future_id", "primary_id"], how="inner")
complete_merged_df.head()

,future_id,47,48,49,50,51,52,53,54,55,...,1779,primary_id,production_cost_total,production_total,emission_avg_last_five_years,emission_total,technical_cost,consumer_savings,human_health,air_pollution
0,1,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,0.190891,...,0.116270,354355,1.484591e+10,7.915055e+08,59.720047,3553.270010,-42.336438,5.511622,7.444526,-33.250075
1,2,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,0.046745,...,0.392241,354356,1.451503e+10,8.012644e+08,30.972144,3028.607700,-21.656242,6.947874,8.685293,-19.779789
2,3,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,0.929365,...,0.402772,354357,1.274256e+10,9.337928e+08,92.551550,4127.536839,-40.157702,6.355524,8.469585,-36.090181
3,4,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,0.946893,...,0.968928,354358,1.235943e+10,8.241354e+08,11.015231,2668.240547,-53.155978,7.441004,8.909666,-6.495959
4,5,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,0.328322,...,0.267191,354359,1.512143e+10,7.863009e+08,41.876734,3343.173022,-77.798140,6.063635,6.719518,-42.784060


In [78]:
# rearrange columns to have future_id and primary_id at the front
cols_order = ["future_id", "primary_id"] + [col for col in complete_merged_df.columns if col not in ["future_id", "primary_id"]]
complete_merged_df = complete_merged_df[cols_order]
complete_merged_df

,future_id,primary_id,47,48,49,50,51,52,53,54,...,1771,1779,production_cost_total,production_total,emission_avg_last_five_years,emission_total,technical_cost,consumer_savings,human_health,air_pollution
0,1,354355,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,...,0.733289,0.116270,1.484591e+10,7.915055e+08,59.720047,3553.270010,-42.336438,5.511622,7.444526,-33.250075
1,2,354356,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,...,0.626129,0.392241,1.451503e+10,8.012644e+08,30.972144,3028.607700,-21.656242,6.947874,8.685293,-19.779789
2,3,354357,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,...,0.467563,0.402772,1.274256e+10,9.337928e+08,92.551550,4127.536839,-40.157702,6.355524,8.469585,-36.090181
3,4,354358,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,...,0.464470,0.968928,1.235943e+10,8.241354e+08,11.015231,2668.240547,-53.155978,7.441004,8.909666,-6.495959
4,5,354359,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,...,0.378535,0.267191,1.512143e+10,7.863009e+08,41.876734,3343.173022,-77.798140,6.063635,6.719518,-42.784060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,996,355350,0.736701,0.546430,0.739303,0.054172,0.451181,0.875108,0.854104,0.601669,...,0.582446,0.800677,1.472198e+10,8.786512e+08,23.599187,2958.134476,-51.800498,8.239335,8.528071,-18.572578
1984,997,355351,0.484244,0.232193,0.411141,0.677399,0.536223,0.275901,0.268436,0.277561,...,0.602409,0.508916,1.331408e+10,8.906465e+08,60.727431,3526.674139,-23.938700,5.442270,6.948918,-19.480756
1985,998,355352,0.594076,0.453653,0.999049,0.260340,0.276599,0.669046,0.335968,0.251319,...,0.138852,0.497848,1.329021e+10,7.832349e+08,51.497226,3434.786875,-129.619974,6.710947,7.131005,-29.458703
1986,999,355353,0.991327,0.185911,0.093308,0.002969,0.027930,0.843634,0.806122,0.438407,...,0.439769,0.640290,1.330764e+10,9.307686e+08,62.552429,3695.679453,-132.245620,6.000841,8.398696,-31.421014


In [80]:
complete_merged_df

,future_id,primary_id,47,48,49,50,51,52,53,54,...,1771,1779,production_cost_total,production_total,emission_avg_last_five_years,emission_total,technical_cost,consumer_savings,human_health,air_pollution
0,1,354355,0.639363,0.261333,0.115895,0.456909,0.389376,0.065254,0.452244,0.395609,...,0.733289,0.116270,1.484591e+10,7.915055e+08,59.720047,3553.270010,-42.336438,5.511622,7.444526,-33.250075
1,2,354356,0.524422,0.141191,0.625291,0.369740,0.475667,0.017979,0.912414,0.894009,...,0.626129,0.392241,1.451503e+10,8.012644e+08,30.972144,3028.607700,-21.656242,6.947874,8.685293,-19.779789
2,3,354357,0.438775,0.902956,0.099015,0.511602,0.405292,0.416929,0.832413,0.406103,...,0.467563,0.402772,1.274256e+10,9.337928e+08,92.551550,4127.536839,-40.157702,6.355524,8.469585,-36.090181
3,4,354358,0.025503,0.833553,0.666840,0.301819,0.569698,0.778987,0.995629,0.994153,...,0.464470,0.968928,1.235943e+10,8.241354e+08,11.015231,2668.240547,-53.155978,7.441004,8.909666,-6.495959
4,5,354359,0.225323,0.718339,0.291168,0.507558,0.499419,0.918567,0.183357,0.397512,...,0.378535,0.267191,1.512143e+10,7.863009e+08,41.876734,3343.173022,-77.798140,6.063635,6.719518,-42.784060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,996,355350,0.736701,0.546430,0.739303,0.054172,0.451181,0.875108,0.854104,0.601669,...,0.582446,0.800677,1.472198e+10,8.786512e+08,23.599187,2958.134476,-51.800498,8.239335,8.528071,-18.572578
1984,997,355351,0.484244,0.232193,0.411141,0.677399,0.536223,0.275901,0.268436,0.277561,...,0.602409,0.508916,1.331408e+10,8.906465e+08,60.727431,3526.674139,-23.938700,5.442270,6.948918,-19.480756
1985,998,355352,0.594076,0.453653,0.999049,0.260340,0.276599,0.669046,0.335968,0.251319,...,0.138852,0.497848,1.329021e+10,7.832349e+08,51.497226,3434.786875,-129.619974,6.710947,7.131005,-29.458703
1986,999,355353,0.991327,0.185911,0.093308,0.002969,0.027930,0.843634,0.806122,0.438407,...,0.439769,0.640290,1.330764e+10,9.307686e+08,62.552429,3695.679453,-132.245620,6.000841,8.398696,-31.421014


## Filter out irrelevant lhs groups

In [81]:
var_traj_X_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_X.csv"))
var_traj_L_df = pd.read_csv(os.path.join(SIMULATION_DIR_PATH, "VARIABLE_TRAJECTORY_GROUPS_L.csv"))

In [53]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
53,elasticity_ippu_wood_production_to_gdp,57
54,elasticity_ippu_product_use_lubricants_product...,58
55,elasticity_ippu_product_use_ods_other_product_...,58
56,elasticity_ippu_product_use_ods_refrigeration_...,58
57,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [ ]:
var_traj_L_df.tail()

In [82]:
var_traj_groups_X = var_traj_X_df["variable_trajectory_group"].unique()
var_traj_groups_L = var_traj_L_df["variable_trajectory_group"].unique()
print("Variable trajectory groups X:", var_traj_groups_X)
print("Variable trajectory groups L:", var_traj_groups_L)

Variable trajectory groups X: [47 48 49 50 51 52 53 54 55 56 57 58]
Variable trajectory groups L: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46]


In [83]:
# join the var_traj_groups
var_traj_groups_all = var_traj_groups_X.tolist() + var_traj_groups_L.tolist()
var_traj_groups_all = list(set(var_traj_groups_all))  # remove duplicates
print("All variable trajectory groups:", var_traj_groups_all)

All variable trajectory groups: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58]


In [84]:
# Convert the variable_trajectory_group column to list of strings
relevant_lhs_cols = [str(col) for col in var_traj_groups_all]

In [85]:
df_cols = complete_merged_df.columns.tolist()

# Filter the relevant_lhs_cols to only include those that are in df_cols
relevant_lhs_cols = [col for col in relevant_lhs_cols if col in df_cols]

In [86]:
# filter complete_merged_df to keep only relevant columns
cols_to_keep = ["future_id", "primary_id"] + list(relevant_lhs_cols) + ["emission_avg_last_five_years", "emission_total", "production_total","production_cost_total","technical_cost", "air_pollution" ,"consumer_savings", "human_health"]
merged_df_filtered = complete_merged_df[cols_to_keep]

In [87]:
print("Original merged DataFrame shape:", complete_merged_df.shape)
print("Filtered merged DataFrame shape:", merged_df_filtered.shape)

Original merged DataFrame shape: (1988, 88)
Filtered merged DataFrame shape: (1988, 68)


In [88]:
print("Filtered merged DataFrame fields:", merged_df_filtered.columns.tolist())
print("Relevant LHS columns:", relevant_lhs_cols)

Filtered merged DataFrame fields: ['future_id', 'primary_id', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', 'emission_avg_last_five_years', 'emission_total', 'production_total', 'production_cost_total', 'technical_cost', 'air_pollution', 'consumer_savings', 'human_health']
Relevant LHS columns: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58']


In [ ]:
merged_df_filtered.head()

## Add variable names to lhs columns

In [89]:
var_traj_L_df.tail()

,transformation_code,variable,variable_field,variable_trajectory_group
470,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_paper,46
471,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_plastic,46
472,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_rubber_leather,46
473,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_textiles,46
474,TX:WASO:INC_RECYCLING,Fraction of Waste Recycled,frac_waso_recycled_wood,46


In [90]:
var_traj_L_df = var_traj_L_df[["variable_field", "variable_trajectory_group"]]
var_traj_L_df = var_traj_L_df.rename(columns={"variable_field": "variable"})
var_traj_L_df.head()

,variable,variable_trajectory_group
0,ef_agrc_anaerobicdom_rice_kg_ch4_ha,1
1,frac_agrc_agriculture_production_lost,2
2,frac_agrc_crop_residues_burned,3
3,frac_agrc_crop_residues_removed,3
4,frac_agrc_no_till_cereals,3


In [91]:
var_traj_X_df.tail()

,variable,variable_trajectory_group
53,elasticity_ippu_wood_production_to_gdp,57
54,elasticity_ippu_product_use_lubricants_product...,58
55,elasticity_ippu_product_use_ods_other_product_...,58
56,elasticity_ippu_product_use_ods_refrigeration_...,58
57,elasticity_ippu_product_use_paraffin_wax_produ...,58


In [92]:
var_traj_all_df = pd.concat([var_traj_X_df, var_traj_L_df], ignore_index=True)
var_traj_all_df

,variable,variable_trajectory_group
0,cost_enfu_fuel_coal_usd_per_tonne,47
1,cost_enfu_fuel_coke_usd_per_tonne,47
2,cost_enfu_fuel_hydrocarbon_gas_liquids_usd_per...,47
3,cost_enfu_fuel_natural_gas_usd_per_mmbtu,47
4,cost_enfu_fuel_crude_usd_per_m3,47
...,...,...
528,frac_waso_recycled_paper,46
529,frac_waso_recycled_plastic,46
530,frac_waso_recycled_rubber_leather,46
531,frac_waso_recycled_textiles,46


In [93]:
# drop duplicates if any
print("Before dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)
var_traj_all_df = var_traj_all_df.drop_duplicates(subset=["variable", "variable_trajectory_group"])
print("After dropping duplicates, var_traj_all_df shape:", var_traj_all_df.shape)

Before dropping duplicates, var_traj_all_df shape: (533, 2)
After dropping duplicates, var_traj_all_df shape: (520, 2)


In [94]:
# check if there are any duplicated variable names
duplicated_vars = var_traj_all_df["variable"].duplicated().any()
if duplicated_vars:
    print("There are duplicated variable names in var_traj_all_df.")
else:
    print("No duplicated variable names in var_traj_all_df.")

No duplicated variable names in var_traj_all_df.


In [95]:
# Filter var_traj_all_df by sample_group in relevant_lhs_cols
relevant_lhs_cols = [int(col) for col in relevant_lhs_cols]
var_traj_all_df = var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin(relevant_lhs_cols)]
var_traj_all_df = var_traj_all_df.sort_values(by="variable_trajectory_group", ascending=True)
print("After filtering by relevant_lhs_cols, var_traj_all_df shape:", var_traj_all_df.shape)

After filtering by relevant_lhs_cols, var_traj_all_df shape: (520, 2)


In [96]:
def process_variable_prefix(df):
    result = []
    for group, group_df in df.groupby('variable_trajectory_group'):
        variables = group_df['variable'].tolist()
        if len(variables) == 1:
            prefix = variables[0]
        else:
            prefix = os.path.commonprefix(variables)
            # Clean trailing underscores
            prefix = prefix.rstrip('_')
            
        prefix = f"group_{group}_{prefix}"
        result.append({'variable_trajectory_group': group, 'variable_prefix': prefix})
    return pd.DataFrame(result)

prefix_df = process_variable_prefix(var_traj_all_df)
prefix_df

,variable_trajectory_group,variable_prefix
0,1,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha
1,2,group_2_frac_agrc_agriculture_production_lost
2,3,group_3_frac_agrc
3,4,group_4_qty_ccsq_mt_co2_captured_sequestered_b...
4,5,group_5_frac_enfu_transmission_loss_fuel_elect...
5,6,group_6_nemomod_en
6,7,group_7_nemomod_entc_frac_min_share_production...
7,8,group_8_frac_fgtv_reduction_in_fugitive_leaks
8,9,group_9_frac_fgtv_drained_and_waste_ch4_flared...
9,10,group_10_efficfactor_enfu_industrial_energy_fuel


In [97]:
# Check for duplicates in variable_trajectory_group and variable_prefix
dups = prefix_df.duplicated(subset=["variable_trajectory_group", "variable_prefix"], keep=False)
if dups.any():
    print("Duplicated variable_trajectory_group and variable_prefix found:")
    print(prefix_df[dups])
else:
    print("No duplicated variable_trajectory_group and variable_prefix found.")

# Check for duplicates in variable_trajectory_group
dups_group = prefix_df.duplicated(subset=["variable_trajectory_group"], keep=False)
if dups_group.any():
    print("Duplicated variable_trajectory_group found:")
    print(prefix_df[dups_group])
else:
    print("No duplicated variable_trajectory_group found.")

# Check for duplicates in variable_prefix
dups_prefix = prefix_df.duplicated(subset=["variable_prefix"], keep=False)
if dups_prefix.any():
    print("Duplicated variable_prefix found:")
    print(prefix_df[dups_prefix])
else:
    print("No duplicated variable_prefix found.")

No duplicated variable_trajectory_group and variable_prefix found.
No duplicated variable_trajectory_group found.
No duplicated variable_prefix found.


In [ ]:
# var_traj_all_df[var_traj_all_df["variable_trajectory_group"].isin([3, 13, 40])]

In [ ]:
# prefix_df.loc[prefix_df["sample_group"] == 13, "variable_prefix"] = "group_13_frac_gnrl_eating_red_meats+"
# prefix_df.loc[prefix_df["sample_group"] == 40, "variable_prefix"] = "group_40_pij_lndu_grasslands+"

# prefix_df = prefix_df.sort_values(by="variable_prefix", ascending=True)
# prefix_df

In [98]:
# Let's use the prefix_df to rename the columns in merged_df_filtered
def rename_columns_with_prefix(merged_df, prefix_df):
    df = merged_df.copy()
    # Create a mapping from str(group) to prefix
    group_to_prefix = {str(row['variable_trajectory_group']): row['variable_prefix'] for _, row in prefix_df.iterrows()}
    # Only rename columns that match a group
    rename_dict = {col: group_to_prefix[col] for col in df.columns if col in group_to_prefix}
    df = df.rename(columns=rename_dict)
    return df

merged_df_filtered_w_prefix = rename_columns_with_prefix(merged_df_filtered, prefix_df)

In [99]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_frac_enfu_transmission_loss_fuel_electricity,group_6_nemomod_en,group_7_nemomod_entc_frac_min_share_production_fp_hydrogen_electrolysis,group_8_frac_fgtv_reduction_in_fugitive_leaks,...,group_57_elasticity_ippu,group_58_elasticity_ippu_product_use,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,consumer_savings,human_health
0,1,354355,0.709094,0.362421,0.352636,0.246076,0.230763,0.231633,0.357253,0.416035,...,0.199758,0.969220,59.720047,3553.270010,7.915055e+08,1.484591e+10,-42.336438,-33.250075,5.511622,7.444526
1,2,354356,0.355127,0.388826,0.495092,0.282101,0.198981,0.620631,0.428909,0.192468,...,0.725034,0.902345,30.972144,3028.607700,8.012644e+08,1.451503e+10,-21.656242,-19.779789,6.947874,8.685293
2,3,354357,0.970473,0.436334,0.243523,0.227546,0.467160,0.102438,0.605842,0.087344,...,0.208350,0.175365,92.551550,4127.536839,9.337928e+08,1.274256e+10,-40.157702,-36.090181,6.355524,8.469585
3,4,354358,0.605323,0.210722,0.661197,0.823838,0.003365,0.990644,0.114855,0.215559,...,0.227383,0.113647,11.015231,2668.240547,8.241354e+08,1.235943e+10,-53.155978,-6.495959,7.441004,8.909666
4,5,354359,0.247035,0.704965,0.523551,0.535350,0.345173,0.482609,0.894310,0.862094,...,0.947292,0.764492,41.876734,3343.173022,7.863009e+08,1.512143e+10,-77.798140,-42.784060,6.063635,6.719518
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,996,355350,0.758811,0.639557,0.801335,0.366066,0.541587,0.853178,0.846512,0.184772,...,0.204425,0.508420,23.599187,2958.134476,8.786512e+08,1.472198e+10,-51.800498,-18.572578,8.239335,8.528071
1984,997,355351,0.038856,0.120206,0.256071,0.181417,0.507199,0.590424,0.975452,0.482633,...,0.826772,0.510035,60.727431,3526.674139,8.906465e+08,1.331408e+10,-23.938700,-19.480756,5.442270,6.948918
1985,998,355352,0.229752,0.060025,0.750683,0.297394,0.263480,0.148370,0.072600,0.332356,...,0.359322,0.344483,51.497226,3434.786875,7.832349e+08,1.329021e+10,-129.619974,-29.458703,6.710947,7.131005
1986,999,355353,0.375548,0.122781,0.293197,0.977969,0.029632,0.088888,0.712697,0.304010,...,0.415388,0.434470,62.552429,3695.679453,9.307686e+08,1.330764e+10,-132.245620,-31.421014,6.000841,8.398696


In [100]:
merged_df_filtered_w_prefix.shape

(1988, 68)

In [101]:
# check for duplicated column names
duplicated_cols = merged_df_filtered_w_prefix.columns[merged_df_filtered_w_prefix.columns.duplicated()].tolist()
if duplicated_cols:
    print("Duplicated column names found:", duplicated_cols)
else:
    print("No duplicated column names found.")

No duplicated column names found.


In [102]:
merged_df_filtered_w_prefix

,future_id,primary_id,group_1_ef_agrc_anaerobicdom_rice_kg_ch4_ha,group_2_frac_agrc_agriculture_production_lost,group_3_frac_agrc,group_4_qty_ccsq_mt_co2_captured_sequestered_by_direct_air_capture,group_5_frac_enfu_transmission_loss_fuel_electricity,group_6_nemomod_en,group_7_nemomod_entc_frac_min_share_production_fp_hydrogen_electrolysis,group_8_frac_fgtv_reduction_in_fugitive_leaks,...,group_57_elasticity_ippu,group_58_elasticity_ippu_product_use,emission_avg_last_five_years,emission_total,production_total,production_cost_total,technical_cost,air_pollution,consumer_savings,human_health
0,1,354355,0.709094,0.362421,0.352636,0.246076,0.230763,0.231633,0.357253,0.416035,...,0.199758,0.969220,59.720047,3553.270010,7.915055e+08,1.484591e+10,-42.336438,-33.250075,5.511622,7.444526
1,2,354356,0.355127,0.388826,0.495092,0.282101,0.198981,0.620631,0.428909,0.192468,...,0.725034,0.902345,30.972144,3028.607700,8.012644e+08,1.451503e+10,-21.656242,-19.779789,6.947874,8.685293
2,3,354357,0.970473,0.436334,0.243523,0.227546,0.467160,0.102438,0.605842,0.087344,...,0.208350,0.175365,92.551550,4127.536839,9.337928e+08,1.274256e+10,-40.157702,-36.090181,6.355524,8.469585
3,4,354358,0.605323,0.210722,0.661197,0.823838,0.003365,0.990644,0.114855,0.215559,...,0.227383,0.113647,11.015231,2668.240547,8.241354e+08,1.235943e+10,-53.155978,-6.495959,7.441004,8.909666
4,5,354359,0.247035,0.704965,0.523551,0.535350,0.345173,0.482609,0.894310,0.862094,...,0.947292,0.764492,41.876734,3343.173022,7.863009e+08,1.512143e+10,-77.798140,-42.784060,6.063635,6.719518
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983,996,355350,0.758811,0.639557,0.801335,0.366066,0.541587,0.853178,0.846512,0.184772,...,0.204425,0.508420,23.599187,2958.134476,8.786512e+08,1.472198e+10,-51.800498,-18.572578,8.239335,8.528071
1984,997,355351,0.038856,0.120206,0.256071,0.181417,0.507199,0.590424,0.975452,0.482633,...,0.826772,0.510035,60.727431,3526.674139,8.906465e+08,1.331408e+10,-23.938700,-19.480756,5.442270,6.948918
1985,998,355352,0.229752,0.060025,0.750683,0.297394,0.263480,0.148370,0.072600,0.332356,...,0.359322,0.344483,51.497226,3434.786875,7.832349e+08,1.329021e+10,-129.619974,-29.458703,6.710947,7.131005
1986,999,355353,0.375548,0.122781,0.293197,0.977969,0.029632,0.088888,0.712697,0.304010,...,0.415388,0.434470,62.552429,3695.679453,9.307686e+08,1.330764e+10,-132.245620,-31.421014,6.000841,8.398696


## Finally we save the processed data as training data

In [103]:
#save the merged DataFrame to a CSV file
merged_df_filtered_w_prefix.to_csv(os.path.join(TRAINING_DIR_PATH, "training_data_v4.2_narrative_profiles_new_ed.csv"), index=False)